# ChemBreak V13 - Colab Enterprise

Enterprise-only controller for the V13 task-bank pipeline.

V13 keeps the V11 research design but fixes Gemini 2.5 Pro structured judging and reduces runtime by batching the three initial Gemini candidates into one model call per assignment.

Start with `RUN_TYPE = "test"`.


## 1. Configure project, repository, and durable Cloud Storage

This notebook uses the existing project bucket that has already been verified as visible from the project. It does not attempt to create a new bucket.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import threading

import google.auth
from google.cloud import storage
from google.api_core.exceptions import NotFound, Forbidden

PROJECT_ID = "rs-foundsecft-mghasemi"
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
PROJECT_SUBDIR = "ChemBreak_V13_Cloud"

# Existing project bucket. No bucket-creation permission is required.
GCS_BUCKET = "rs-foundsecft-mghasemi-default-1"
GCS_PREFIX = "ChemBreak_V13"
GCS_SYNC_SECONDS = 60

credentials, adc_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

print("ADC detected:", type(credentials).__name__)
print("ADC project:", adc_project)
print("Quota project:", PROJECT_ID)

RUNTIME_ROOT = Path("/content/chembreak_v13_runtime")
REPO_ROOT = RUNTIME_ROOT / "ChemBreak_repo"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in the repository. "
        "Upload the V13 folder to GitHub first."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v13_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

pipeline_source = PIPELINE.read_text(encoding="utf-8")
required_markers = [
    'VERSION = "13.0-cloud"',
    'NAMESPACE = "CBV13C"',
    'responseSchema',
    '_generator_batch_schema',
    '_judge_response_schema',
    '_score_array_to_dict',
]
missing = [m for m in required_markers if m not in pipeline_source]
if missing:
    raise RuntimeError(f"Wrong or incomplete V13 pipeline loaded: {missing}")

print("Project directory:", PROJECT_DIR)
print("V13 pipeline verification: PASSED")


## 2. Verify access to the existing bucket

V13 only needs object read/write access. The notebook does not create or administer buckets.


In [ ]:
storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)

try:
    bucket = storage_client.get_bucket(GCS_BUCKET)
except (NotFound, Forbidden) as exc:
    raise RuntimeError(
        f"Cannot access gs://{GCS_BUCKET}. Use an existing bucket with object read/write access."
    ) from exc

# Small object-level write/read/delete preflight.
probe_name = f"{GCS_PREFIX}/_access_probe.txt"
probe = bucket.blob(probe_name)
try:
    probe.upload_from_string("ChemBreak V13 access probe")
    probe.download_as_text()
    probe.delete()
except Forbidden as exc:
    raise RuntimeError(
        f"The bucket is visible but this account cannot create/read/delete objects under "
        f"gs://{GCS_BUCKET}/{GCS_PREFIX}/"
    ) from exc

print("Bucket access: PASSED")
print(f"Durable root: gs://{GCS_BUCKET}/{GCS_PREFIX}/")


## 3. Select run type and restore an existing V13 checkpoint

Each run type has its own durable prefix. V13 never mixes with V11 outputs.


In [ ]:
RUN_TYPE = "test"   # test | pilot | production

LOCAL_OUTPUT_DIR = RUNTIME_ROOT / "outputs" / RUN_TYPE
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GCS_OUTPUT_PREFIX = f"{GCS_PREFIX}/outputs/{RUN_TYPE}"

RUNTIME_CONFIG = RUNTIME_ROOT / f"chembreak_v13_{RUN_TYPE}.json"
cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["project_id"] = PROJECT_ID
cfg["run_type"] = RUN_TYPE
RUNTIME_CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

_uploaded_fingerprints = {}

def _stable_bytes(path: Path):
    try:
        s1 = path.stat()
        data = path.read_bytes()
        s2 = path.stat()
    except FileNotFoundError:
        return None, None
    fp1 = (s1.st_size, s1.st_mtime_ns)
    fp2 = (s2.st_size, s2.st_mtime_ns)
    if fp1 != fp2:
        return None, None
    return data, fp2

def sync_to_gcs(verbose=True):
    uploaded = 0
    for path in LOCAL_OUTPUT_DIR.rglob("*"):
        if not path.is_file():
            continue
        data, fp = _stable_bytes(path)
        if data is None:
            continue
        rel = path.relative_to(LOCAL_OUTPUT_DIR).as_posix()
        if _uploaded_fingerprints.get(rel) == fp:
            continue
        bucket.blob(f"{GCS_OUTPUT_PREFIX}/{rel}").upload_from_string(data)
        _uploaded_fingerprints[rel] = fp
        uploaded += 1
    if verbose:
        print(
            f"GCS sync complete: {uploaded} changed file(s) -> "
            f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/"
        )
    return uploaded

def restore_from_gcs():
    prefix = f"{GCS_OUTPUT_PREFIX}/"
    restored = 0
    for blob in storage_client.list_blobs(GCS_BUCKET, prefix=prefix):
        rel = blob.name[len(prefix):]
        if not rel or rel.endswith("/"):
            continue
        target = LOCAL_OUTPUT_DIR / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        blob.download_to_filename(str(target))
        restored += 1
    print(
        f"Checkpoint restore: {restored} file(s) <- "
        f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/"
    )
    return restored

restore_from_gcs()
print("Run type:", RUN_TYPE)
print("Local working output:", LOCAL_OUTPUT_DIR)
print("Durable output:", f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/")


## 4. Live stage runner with durable checkpoint mirroring


In [ ]:
def run_stage(stage):
    command = [
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(LOCAL_OUTPUT_DIR),
    ]

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    env["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

    stop_sync = threading.Event()

    def _mirror_loop():
        while not stop_sync.wait(GCS_SYNC_SECONDS):
            try:
                changed = sync_to_gcs(verbose=False)
                if changed:
                    print(f"\n[V13 GCS CHECKPOINT] mirrored {changed} changed file(s)", flush=True)
            except Exception as exc:
                print(f"\n[V13 GCS CHECKPOINT WARNING] {type(exc).__name__}: {exc}", flush=True)

    mirror_thread = threading.Thread(target=_mirror_loop, daemon=True)
    started = time.time()
    print(f"\n===== V13 {stage.upper()} START =====", flush=True)
    print("Durable checkpoint:", f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/", flush=True)
    mirror_thread.start()

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
    finally:
        process.stdout.close()
        return_code = process.wait()
        stop_sync.set()
        mirror_thread.join(timeout=10)
        try:
            sync_to_gcs(verbose=True)
        except Exception as exc:
            print(f"[V13 FINAL GCS SYNC WARNING] {type(exc).__name__}: {exc}", flush=True)

    elapsed = time.time() - started
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"===== V13 {stage.upper()} DONE | elapsed {elapsed/60:.1f} min =====\n", flush=True)

print("V13 Enterprise stage runner ready.")


## 5. Preflight


In [ ]:
run_stage("preflight")


## 6. Bootstrap source data


In [ ]:
run_stage("bootstrap")


## 7. Create the V13 assignment plan


In [ ]:
run_stage("plan")


## 8. Generate the candidate pool

V13 normally uses one structured Gemini 3.1 call per assignment to return the requested A/B/C candidates together, rather than three separate calls.


In [ ]:
run_stage("generate")


## 9. Deterministic validation


In [ ]:
run_stage("validate")


## 10. Repair invalid candidates


In [ ]:
run_stage("repair")


## 11. Pre-judge recovery


In [ ]:
run_stage("prejudge_refill")


## 12. Two independent judges

gpt-oss-120B and Gemini 2.5 Pro judge the same active candidate set independently. Gemini 2.5 Pro uses an explicit Vertex AI response schema in V13.


In [ ]:
run_stage("judge")


## 13. Blind adjudication


In [ ]:
run_stage("adjudicate")


## 14. Full refill cycles


In [ ]:
for cycle in range(int(cfg["recovery"]["max_full_refill_cycles"])):
    print(f"\n===== V13 FULL REFILL CYCLE {cycle + 1} =====")
    run_stage("refill")
    run_stage("prejudge_refill")
    run_stage("judge")
    run_stage("adjudicate")


## 15. Finalize the task bank


In [ ]:
run_stage("finalize")


## 16. Final status and sync


In [ ]:
run_stage("status")
sync_to_gcs(verbose=True)
print("\nV13 Enterprise run complete.")
print("Durable results:", f"gs://{GCS_BUCKET}/{GCS_OUTPUT_PREFIX}/")


## Resume after a runtime restart

Reconnect to Colab Enterprise and rerun Sections 1-4. Section 3 restores the V13 checkpoint from Cloud Storage. Then rerun the interrupted stage. Completed work is skipped by the pipeline checkpoints and run-compatibility guard.
